# Ablasyon Çalışması — ST-GCN NTU RGB+D 60 XSub

Her deney **40 epoch** çalışır (trend görmek için yeterli).  
Kazanan config → `02_model_training.ipynb`'de tam 80 epoch ile eğitilir.

| # | Ablasyon | Değişken |
|---|----------|----------|
| A1 | Normalizasyon stratejisi | raw / center-only / full-norm (baseline) |
| A2 | Temporal çözünürlük | T=50 / T=75 / T=100 (baseline) |
| A3 | Adjacency matrix | Uniform / Distance / Spatial-Config |
| A4 | Model mimarisi | ST-GCN / ST-GCN++ (multi-scale TCN) |

**Drive'a yüklenmiş olması gerekenler:**
- `MyDrive/stgcn/processed_xsub.npz`
- `MyDrive/stgcn/ntu60_hrnet.pkl` (A1 normalizasyon ablasyonu için)

## 0. Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

DRIVE_ROOT = '/content/drive/MyDrive/stgcn'
OUT_DIR    = os.path.join(DRIVE_ROOT, 'ablation_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

NPZ_PATH = os.path.join(DRIVE_ROOT, 'processed_xsub.npz')
PKL_PATH = os.path.join(DRIVE_ROOT, 'ntu60_hrnet.pkl')

## 1. Ortak Bileşenler
Tüm deneylerde kullanılacak model, dataset ve eğitim fonksiyonları.

In [ ]:
# ── Adjacency matrix builder'ları ──────────────────────────────────────────

COCO_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]
NUM_JOINTS = 17

def build_adj_uniform(num_joints=NUM_JOINTS, pairs=COCO_PAIRS):
    """Tüm komşular eşit ağırlık — D^{-1/2} A D^{-1/2}"""
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    for i, j in pairs:
        A[i, j] = A[j, i] = 1.0
    A += np.eye(num_joints, dtype=np.float32)
    D = np.diag(A.sum(1) ** -0.5)
    return torch.FloatTensor(D @ A @ D)

def build_adj_distance(num_joints=NUM_JOINTS, pairs=COCO_PAIRS, max_hop=1):
    """Mesafeye göre ağırlık — uzak komşular daha düşük ağırlık"""
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    for i, j in pairs:
        A[i, j] = A[j, i] = 1.0
    # 2-hop komşuları ekle (daha düşük ağırlıkla)
    A2 = np.linalg.matrix_power((A > 0).astype(float), 2)
    A_dist = A.copy()
    for i in range(num_joints):
        for j in range(num_joints):
            if A2[i, j] > 0 and A[i, j] == 0 and i != j:
                A_dist[i, j] = 0.5
    A_dist += np.eye(num_joints, dtype=np.float32)
    D = np.diag(A_dist.sum(1) ** -0.5)
    return torch.FloatTensor(D @ A_dist @ D)

def build_adj_spatial_config(num_joints=NUM_JOINTS, pairs=COCO_PAIRS):
    """
    ST-GCN paper'ın Spatial Configuration Partitioning'i.
    Her joint için komşuları 3 gruba ayır:
    - Centripetal: gravity center'a yakın komşular  
    - Centrifugal: gravity center'dan uzak komşular
    - Root: aynı mesafedekiler
    3 ayrı adjacency matrisi döndürür → model bunları ayrı ayrı işler.
    """
    A_base = np.zeros((num_joints, num_joints), dtype=np.float32)
    for i, j in pairs:
        A_base[i, j] = A_base[j, i] = 1.0

    # Gravity center: tüm joint'lerin ağırlık merkezi
    # Joint'lere BFS ile gravity center'a mesafeyi hesapla
    center = 11  # sol kalça — yaklaşık kütle merkezi
    dist = np.full(num_joints, np.inf)
    dist[center] = 0
    queue = [center]
    while queue:
        node = queue.pop(0)
        for nbr in range(num_joints):
            if A_base[node, nbr] > 0 and dist[nbr] == np.inf:
                dist[nbr] = dist[node] + 1
                queue.append(nbr)

    A_root       = np.zeros_like(A_base)
    A_centripetal = np.zeros_like(A_base)
    A_centrifugal = np.zeros_like(A_base)

    for i in range(num_joints):
        A_root[i, i] = 1.0  # self-loop
        for j in range(num_joints):
            if A_base[i, j] > 0:
                if dist[j] < dist[i]:
                    A_centripetal[i, j] = 1.0
                elif dist[j] > dist[i]:
                    A_centrifugal[i, j] = 1.0
                else:
                    A_root[i, j] = 1.0

    def norm(M):
        M = M.astype(np.float32)
        row_sum = M.sum(1, keepdims=True)
        row_sum[row_sum == 0] = 1
        return M / row_sum

    return [torch.FloatTensor(norm(A_root)),
            torch.FloatTensor(norm(A_centripetal)),
            torch.FloatTensor(norm(A_centrifugal))]

print("Adjacency builder'lar hazır.")
print(f"  Uniform non-zeros   : {(build_adj_uniform() > 0).sum().item()}")
print(f"  Distance non-zeros  : {(build_adj_distance() > 0).sum().item()}")
sc = build_adj_spatial_config()
print(f"  SpatialConfig parts : {len(sc)} matris")

In [ ]:
# ── Model tanımları ────────────────────────────────────────────────────────

class SpatialGCN(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        if isinstance(A, list):  # Spatial Config: 3 matris
            self.num_A = len(A)
            for i, a in enumerate(A):
                self.register_buffer(f'A{i}', a)
            self.conv = nn.Conv2d(in_c * self.num_A, out_c, kernel_size=1)
        else:
            self.num_A = 1
            self.register_buffer('A0', A)
            self.conv = nn.Conv2d(in_c, out_c, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_c)

    def forward(self, x):
        if self.num_A == 1:
            x = torch.einsum('nctv,vw->nctw', x, self.A0)
        else:
            parts = [torch.einsum('nctv,vw->nctw', x, getattr(self, f'A{i}'))
                     for i in range(self.num_A)]
            x = torch.cat(parts, dim=1)  # channel boyutunda birleştir
        return self.bn(self.conv(x))


class STGCNBlock(nn.Module):
    """Standart ST-GCN bloğu (tek ölçekli temporal conv)"""
    def __init__(self, in_c, out_c, A, stride=1):
        super().__init__()
        self.gcn = SpatialGCN(in_c, out_c, A)
        self.tcn = nn.Sequential(
            nn.Conv2d(out_c, out_c, (9,1), stride=(stride,1), padding=(4,0)),
            nn.BatchNorm2d(out_c),
        )
        self.residual = (
            nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=(stride,1)),
                          nn.BatchNorm2d(out_c))
            if in_c != out_c or stride != 1 else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.tcn(self.relu(self.gcn(x))) + self.residual(x))


class MultiScaleTCN(nn.Module):
    """ST-GCN++ tarzı çok ölçekli temporal conv (3 dal)"""
    def __init__(self, channels, stride=1):
        super().__init__()
        c = channels // 4
        # Dal 1: kısa (kernel=3)
        self.br1 = nn.Sequential(
            nn.Conv2d(channels, c, (3,1), stride=(stride,1), padding=(1,0)),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True)
        )
        # Dal 2: orta (kernel=9)
        self.br2 = nn.Sequential(
            nn.Conv2d(channels, c, (9,1), stride=(stride,1), padding=(4,0)),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True)
        )
        # Dal 3: dilated (kernel=9, dilation=2)
        self.br3 = nn.Sequential(
            nn.Conv2d(channels, c, (9,1), stride=(stride,1), padding=(8,0), dilation=(2,1)),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True)
        )
        # Dal 4: stride-only (MaxPool)
        self.br4 = nn.Sequential(
            nn.MaxPool2d((3,1), stride=(stride,1), padding=(1,0)),
            nn.BatchNorm2d(channels)
        )
        # Projection: 3c → channels
        self.proj = nn.Sequential(
            nn.Conv2d(3*c + channels, channels, 1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        return self.proj(torch.cat([self.br1(x), self.br2(x), self.br3(x), self.br4(x)], dim=1))


class STGCNPlusBlock(nn.Module):
    """ST-GCN++ bloğu: aynı spatial GCN, çok ölçekli temporal"""
    def __init__(self, in_c, out_c, A, stride=1):
        super().__init__()
        self.gcn = SpatialGCN(in_c, out_c, A)
        self.tcn = MultiScaleTCN(out_c, stride=stride)
        self.residual = (
            nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=(stride,1)),
                          nn.BatchNorm2d(out_c))
            if in_c != out_c or stride != 1 else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.tcn(self.relu(self.gcn(x))) + self.residual(x))


def build_stgcn(A, block_cls=STGCNBlock, num_classes=60, in_channels=2):
    """A: tek Tensor (uniform/distance) veya list (spatial-config)"""
    layers = nn.ModuleList([
        block_cls(in_channels, 64,  A),
        block_cls(64,  64,  A),
        block_cls(64,  64,  A),
        block_cls(64,  64,  A),
        block_cls(64,  128, A, stride=2),
        block_cls(128, 128, A),
        block_cls(128, 128, A),
        block_cls(128, 256, A, stride=2),
        block_cls(256, 256, A),
        block_cls(256, 256, A),
    ])

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.data_bn = nn.BatchNorm1d(in_channels * NUM_JOINTS)
            self.layers  = layers
            self.pool    = nn.AdaptiveAvgPool2d(1)
            self.drop    = nn.Dropout(0.5)
            self.fc      = nn.Linear(256, num_classes)
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode='fan_out')
                elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)

        def forward(self, x):
            N, C, T, V, M = x.shape
            x = x.permute(0,4,1,3,2).contiguous().view(N*M, C*V, T)
            x = self.data_bn(x)
            x = x.view(N*M, C, V, T).permute(0,1,3,2)
            for layer in self.layers:
                x = layer(x)
            x = self.pool(x).view(N*M, -1)
            x = x.view(N, M, -1).mean(1)
            return self.fc(self.drop(x))

    return Model()

# Parametre sayısı kontrolü
A_uni = build_adj_uniform()
m_base = build_stgcn(A_uni)
m_plus = build_stgcn(A_uni, block_cls=STGCNPlusBlock)
print(f"ST-GCN  params: {sum(p.numel() for p in m_base.parameters())/1e6:.2f}M")
print(f"ST-GCN+ params: {sum(p.numel() for p in m_plus.parameters())/1e6:.2f}M")

In [ ]:
# ── Dataset & eğitim fonksiyonları ─────────────────────────────────────────

class SkeletonDataset(Dataset):
    def __init__(self, X, y, target_t=None, augment=False):
        """
        X: (N, M, T, V, C)
        target_t: None = olduğu gibi kullan, int = T boyutunu yeniden sample et
        """
        if target_t is not None and target_t != X.shape[2]:
            T = X.shape[2]
            idx = np.linspace(0, T-1, target_t, dtype=int)
            X = X[:, :, idx, :, :]
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.augment = augment

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]  # (M, T, V, C)
        if self.augment:
            angle = (torch.rand(1).item() - 0.5) * 60 * (3.14159/180)
            R = torch.tensor([[np.cos(angle), -np.sin(angle)],
                               [np.sin(angle),  np.cos(angle)]], dtype=torch.float32)
            x = torch.einsum('mtvc,cd->mtvd', x, R)
            x = x * (0.9 + torch.rand(1).item() * 0.2)
        return x.permute(3,1,2,0).contiguous(), self.y[idx]  # (C,T,V,M)


def make_loaders(X_train, y_train, X_val, y_val, target_t=None, batch=64):
    tr = DataLoader(SkeletonDataset(X_train, y_train, target_t, augment=True),
                    batch_size=batch, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
    va = DataLoader(SkeletonDataset(X_val, y_val, target_t, augment=False),
                    batch_size=batch, shuffle=False, num_workers=4, pin_memory=True)
    return tr, va


def train_experiment(model, train_loader, val_loader, name, epochs=40):
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1,
                                momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler()

    best_acc = 0.0
    history  = defaultdict(list)

    print(f"\n{'='*60}")
    print(f"  Deney: {name}")
    print(f"  Parametre: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    print(f"{'='*60}")
    print(f"{'Ep':>3}  {'TrLoss':>7}  {'TrAcc':>6}  {'VaLoss':>7}  {'VaAcc':>6}  {'t':>5}")

    for ep in range(1, epochs+1):
        t0 = time.time()
        for phase, loader, train_mode in [('tr', train_loader, True), ('va', val_loader, False)]:
            model.train() if train_mode else model.eval()
            loss_sum, correct, total = 0.0, 0, 0
            ctx = torch.enable_grad() if train_mode else torch.no_grad()
            with ctx:
                for x, y in loader:
                    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                    if train_mode:
                        optimizer.zero_grad(set_to_none=True)
                    with autocast():
                        logits = model(x)
                        loss   = criterion(logits, y)
                    if train_mode:
                        scaler.scale(loss).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    loss_sum += loss.item() * y.size(0)
                    correct  += (logits.argmax(1) == y).sum().item()
                    total    += y.size(0)
            history[f'{phase}_loss'].append(loss_sum / total)
            history[f'{phase}_acc'].append(correct / total)

        scheduler.step()
        va_acc = history['va_acc'][-1]
        if va_acc > best_acc:
            best_acc = va_acc

        if ep % 5 == 0 or ep == epochs:
            print(f"{ep:3d}  "
                  f"{history['tr_loss'][-1]:7.4f}  "
                  f"{history['tr_acc'][-1]*100:5.1f}%  "
                  f"{history['va_loss'][-1]:7.4f}  "
                  f"{history['va_acc'][-1]*100:5.1f}%  "
                  f"{time.time()-t0:4.1f}s")

    print(f"  → En iyi val acc: {best_acc*100:.2f}%")
    torch.save({'history': dict(history), 'best_acc': best_acc},
               os.path.join(OUT_DIR, f'{name.replace(" ","_")}.pth'))
    return best_acc, dict(history)

In [ ]:
# Baseline veriyi yükle (tüm deneyler için)
print("Veri yükleniyor...")
d = np.load(NPZ_PATH)
X_train, y_train = d['X_train'], d['y_train']
X_val,   y_val   = d['X_val'],   d['y_val']
print(f"X_train: {X_train.shape}   X_val: {X_val.shape}")

## A1 — Normalizasyon Stratejisi

Hangi preprocessing adımının accuracy'e ne kadar katkı sağladığını ölçüyoruz.

| Config | Açıklama |
|--------|----------|
| raw | Hiç normalizasyon yok (piksel koordinatları) |
| center-only | Sadece kalça merkezine öteleme |
| full-norm | Öteleme + torso scale normalize (baseline) |

In [ ]:
LEFT_HIP, RIGHT_HIP   = 11, 12
LEFT_SH,  RIGHT_SH    = 5,  6

def load_and_preprocess(pkl_path, norm_mode='full'):
    """
    norm_mode: 'raw' | 'center' | 'full'
    Dönüş: X_train (N,M,100,17,2), y_train, X_val, y_val
    """
    with open(pkl_path, 'rb') as f:
        raw = pickle.load(f)
    anns  = raw['annotations']
    split = raw['split']
    fd_map = {a['frame_dir']: a for a in anns}

    def process(fd_list):
        X_list, y_list = [], []
        for fd in fd_list:
            if fd not in fd_map: continue
            ann = fd_map[fd]
            kp  = ann['keypoint'].astype(np.float32)  # (M,T,17,2)
            M, T, V, C = kp.shape

            if norm_mode in ('center', 'full'):
                hip_c = (kp[:,:,LEFT_HIP,:] + kp[:,:,RIGHT_HIP,:]) / 2.0
                kp   -= hip_c[:,:,np.newaxis,:]
            if norm_mode == 'full':
                sh_c       = (kp[:,:,LEFT_SH,:] + kp[:,:,RIGHT_SH,:]) / 2.0
                scale      = np.maximum(np.linalg.norm(sh_c, axis=-1, keepdims=True), 1e-6)
                kp        /= scale[:,:,:,np.newaxis]

            # T=100'e pad/sample
            if T < 100:
                kp = np.concatenate([kp, np.zeros((M,100-T,V,C), np.float32)], axis=1)
            elif T > 100:
                kp = kp[:, np.linspace(0,T-1,100,dtype=int), :, :]

            # Max 2 kişi
            if M < 2:
                kp = np.concatenate([kp, np.zeros((2-M,100,V,C), np.float32)], axis=0)
            else:
                kp = kp[:2]

            X_list.append(kp)
            y_list.append(ann['label'])
        return np.stack(X_list), np.array(y_list)

    print(f"  [{norm_mode}] train işleniyor...")
    Xtr, ytr = process(split['xsub_train'])
    print(f"  [{norm_mode}] val işleniyor...")
    Xva, yva = process(split['xsub_val'])
    return Xtr, ytr, Xva, yva

print("PKL path var mı:", os.path.exists(PKL_PATH))

In [ ]:
A_uni  = build_adj_uniform()
a1_results = {}

for norm_mode in ['raw', 'center', 'full']:
    Xtr, ytr, Xva, yva = load_and_preprocess(PKL_PATH, norm_mode)
    tr_ldr, va_ldr = make_loaders(Xtr, ytr, Xva, yva)
    model = build_stgcn(A_uni).to(device)
    best_acc, hist = train_experiment(model, tr_ldr, va_ldr, f'A1_{norm_mode}', epochs=40)
    a1_results[norm_mode] = best_acc

print("\n=== A1 Özet ===")
for k, v in a1_results.items():
    print(f"  {k:12s}: {v*100:.2f}%")

## A2 — Temporal Çözünürlük

Daha az frame ile ne kadar bilgi kaybediyoruz?

| Config | Açıklama |
|--------|----------|
| T=50 | 100 frame'den her 2. frame alınır |
| T=75 | 100 frame'den uniform subsample |
| T=100 | Tam veri (baseline) |

In [ ]:
a2_results = {}

for T in [50, 75, 100]:
    tr_ldr, va_ldr = make_loaders(X_train, y_train, X_val, y_val, target_t=T)
    model = build_stgcn(A_uni).to(device)
    best_acc, hist = train_experiment(model, tr_ldr, va_ldr, f'A2_T{T}', epochs=40)
    a2_results[f'T={T}'] = best_acc

print("\n=== A2 Özet ===")
for k, v in a2_results.items():
    print(f"  {k:8s}: {v*100:.2f}%")

## A3 — Adjacency Matrix Stratejisi

Graph yapısının modelin öğrenmesine etkisi.

| Config | Açıklama |
|--------|----------|
| Uniform | Tüm komşular eşit ağırlık |
| Distance | 2-hop komşular daha düşük ağırlık |
| SpatialConfig | ST-GCN paper'ın orijinal stratejisi (3 matrisle) |

In [ ]:
tr_ldr, va_ldr = make_loaders(X_train, y_train, X_val, y_val)
a3_results = {}

adj_configs = {
    'Uniform'       : build_adj_uniform(),
    'Distance'      : build_adj_distance(),
    'SpatialConfig' : build_adj_spatial_config(),
}

for name, A in adj_configs.items():
    model = build_stgcn(A).to(device)
    best_acc, hist = train_experiment(model, tr_ldr, va_ldr, f'A3_{name}', epochs=40)
    a3_results[name] = best_acc

print("\n=== A3 Özet ===")
for k, v in a3_results.items():
    print(f"  {k:15s}: {v*100:.2f}%")

## A4 — Model Mimarisi: ST-GCN vs ST-GCN++

Multi-scale temporal conv'un katkısı ne kadar?

| Config | Temporal Conv |
|--------|---------------|
| ST-GCN | Tek ölçek (kernel=9) |
| ST-GCN++ | 3 dal: kernel=3, 9, 9-dilated + MaxPool |

In [ ]:
tr_ldr, va_ldr = make_loaders(X_train, y_train, X_val, y_val)
a4_results = {}

# En iyi A3 config'ini kullan (veya default uniform)
best_adj_name = max(a3_results, key=a3_results.get)
best_A = adj_configs[best_adj_name]
print(f"A4 için adjacency: {best_adj_name}")

for name, block_cls in [('ST-GCN', STGCNBlock), ('ST-GCN++', STGCNPlusBlock)]:
    model = build_stgcn(best_A, block_cls=block_cls).to(device)
    best_acc, hist = train_experiment(model, tr_ldr, va_ldr, f'A4_{name}', epochs=40)
    a4_results[name] = best_acc

print("\n=== A4 Özet ===")
for k, v in a4_results.items():
    print(f"  {k:10s}: {v*100:.2f}%")

## Sonuç Karşılaştırması

In [ ]:
all_results = {
    'A1 — Normalizasyon': a1_results,
    'A2 — Temporal T'   : a2_results,
    'A3 — Adjacency'    : a3_results,
    'A4 — Mimari'       : a4_results,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
colors = ['#d9534f', '#f0ad4e', '#5cb85c', '#5bc0de']

for ax, (title, results), color in zip(axes, all_results.items(), colors):
    names  = list(results.keys())
    values = [v * 100 for v in results.values()]
    bars   = ax.bar(names, values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Val Accuracy (%)')
    ax.set_ylim(max(0, min(values)-5), min(100, max(values)+5))
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Ablasyon Sonuçları — ST-GCN NTU60 XSub (40 epoch)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'ablation_summary.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Rapor için özet tablo
print("=" * 55)
print(f"{'Deney':<30} {'Config':<15} {'Acc':>6}")
print("=" * 55)
for group, results in all_results.items():
    best_cfg = max(results, key=results.get)
    for cfg, acc in results.items():
        marker = " ◄ best" if cfg == best_cfg else ""
        print(f"  {group:<28} {cfg:<15} {acc*100:5.2f}%{marker}")
    print()
print("=" * 55)
print("\nLiteratür baseline: ~85% (PYSKL ST-GCN, 2D HRNet, NTU60-XSub)")

In [ ]:
# Kazanan config'i belirle → 02_model_training.ipynb'de kullanılacak
best_norm = max(a1_results, key=a1_results.get)
best_T    = int(max(a2_results, key=a2_results.get).split('=')[1])
best_adj  = max(a3_results, key=a3_results.get)
best_arch = max(a4_results, key=a4_results.get)

print("\n🏆 Kazanan Konfigürasyon (tam eğitimde kullanılacak)")
print(f"   Normalizasyon : {best_norm}")
print(f"   Temporal T    : {best_T}")
print(f"   Adjacency     : {best_adj}")
print(f"   Mimari        : {best_arch}")
print("\n→ Bu config ile 02_model_training.ipynb'i 80 epoch çalıştır.")